In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import ast
import pprint
from tqdm.auto import tqdm
from api_utils import Utils
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

In [2]:
import os
import requests
from zipfile import ZipFile

data_dir = "./data"
zip_path = os.path.join(data_dir, "lesson2-wiki.csv.zip")
os.makedirs(data_dir, exist_ok=True)

# Download
if not os.path.exists(zip_path):
    print("Downloading dataset...")
    url = "https://www.dropbox.com/scl/fi/yxzmsrv2sgl249zcspeqb/lesson2-wiki.csv.zip?rlkey=paehnoxjl3s5x53d1bedt4pmc&dl=1"
    response = requests.get(url)
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        f.write(response.content)
    print("Download complete!")
else:
    print("Dataset zip already exists, skipping download.")

# Extract
print("Extracting files...")
with ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(data_dir)

print("Done! Dataset is in ./data/")

Dataset zip already exists, skipping download.
Extracting files...
Done! Dataset is in ./data/


# Dataset

In [3]:
df = pd.read_csv('./data/wiki.csv')
df

,id,metadata,values
1,1-0,"{'chunk': 0, 'source': 'https://simple.wikiped...","[-0.011254455894231796, -0.01698738895356655, ..."
2,1-1,"{'chunk': 1, 'source': 'https://simple.wikiped...","[-0.0015197008615359664, -0.007858820259571075..."
3,1-2,"{'chunk': 2, 'source': 'https://simple.wikiped...","[-0.009930099360644817, -0.012211072258651257,..."
4,1-3,"{'chunk': 3, 'source': 'https://simple.wikiped...","[-0.011600767262279987, -0.012608098797500134,..."
5,1-4,"{'chunk': 4, 'source': 'https://simple.wikiped...","[-0.026462381705641747, -0.016362832859158516,..."
...,...,...,...
9996,9273-14,"{'chunk': 14, 'source': 'https://simple.wikipe...","[0.006269281730055809, -0.007062565069645643, ..."
9997,9273-15,"{'chunk': 15, 'source': 'https://simple.wikipe...","[-0.007164978422224522, -0.0002860440290533006..."
9998,9273-16,"{'chunk': 16, 'source': 'https://simple.wikipe...","[0.001473232638090849, -0.024397650733590126, ..."
9999,9273-17,"{'chunk': 17, 'source': 'https://simple.wikipe...","[0.004176019225269556, -0.022336846217513084, ..."


### Embedding

In [4]:
value_0 = df['values'].iloc[0]
print(f'Type of "values" before casting: {type(value_0)}')

if isinstance(value_0, str):
    value_0 = ast.literal_eval(value_0)
print(f'Type of "values" after casting: {type(value_0)}')

EMBEDDING_DIM = len(value_0)
print(f'Embedding dim of each sequence: {len(value_0)}')

Type of "values" before casting: <class 'str'>
Type of "values" after casting: <class 'list'>
Embedding dim of each sequence: 1536


In [5]:
def parse(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

In [6]:
def reduce_embeddings(df, target_dim=384, in_place=True):
    if not in_place:
        df = df.copy()

    # convert each row to list
    df['values'] = df['values'].apply(parse)

    # convert to array from list[list[float]]
    embeddings = np.array(df['values'].tolist(), dtype='float32')
    print(f'Original embedding dim: {embeddings.shape[1]}')

    # model
    pca = PCA(n_components=target_dim)
    embeddings_reduced = pca.fit_transform(embeddings)
    print(f'Embedding dim after reduction: {embeddings_reduced.shape[1]}')

    variance_retained = np.sum(pca.explained_variance_ratio_)
    print(f'Information retained: {variance_retained:.4f} ({variance_retained*100:.2f}%)')

    # store in df
    df['values_reduced'] = [i.tolist() for i in embeddings_reduced]
    return df

In [7]:
REDUCED_EMBEDDING_DIM = 768

if 'values_reduced' not in df.columns:
    reduce_embeddings(df, target_dim=REDUCED_EMBEDDING_DIM)
else:
    print('The embedding size alread reduced')

Original embedding dim: 1536
Embedding dim after reduction: 768
Information retained: 0.9689 (96.89%)


- **I reduce the size of embeddings because I will use different embedding model to encode question (which is different in size)**
- **The embedding model for dataset is from Open AI, but I cannot access to it to encode the question, so I need to reduce size to match with current embedding model for question**

### Metadata

In [8]:
meta_0 = df.metadata.iloc[0]
print(meta_0)
print("~"*100)
print(f'Type of "metadata" before casting: {type(meta_0)}')

{'chunk': 0, 'source': 'https://simple.wikipedia.org/wiki/April', 'text': "April is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of week as July, and additionally, January in leap years. April always ends on the same day of the week as December.\n\nApril's flowers are the Sweet Pea and Daisy. Its birthstone is the diamond. The meaning of the diamond is innocence.\n\nThe Month \n\nApril comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.\n\nApril begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.\n\nIn common years, April s

In [9]:
if isinstance(meta_0, str):
    meta_0 = ast.literal_eval(meta_0)
print(f'Type of "metadata" after casting: {type(meta_0)}')
print("~"*100)
pprint.pprint(meta_0)

Type of "metadata" after casting: <class 'dict'>
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
{'chunk': 0,
 'source': 'https://simple.wikipedia.org/wiki/April',
 'text': 'April is the fourth month of the year in the Julian and Gregorian '
         'calendars, and comes between March and May. It is one of four months '
         'to have 30 days.\n'
         '\n'
         'April always begins on the same day of week as July, and '
         'additionally, January in leap years. April always ends on the same '
         'day of the week as December.\n'
         '\n'
         "April's flowers are the Sweet Pea and Daisy. Its birthstone is the "
         'diamond. The meaning of the diamond is innocence.\n'
         '\n'
         'The Month \n'
         '\n'
         'April comes between March and May, making it the fourth month of the '
         'year. It also comes first in the year out of the four months that '
         'have 30 days,

# Upsert to Vector DB

In [10]:
index = Utils.create_index(index_name='second-index', dimension=REDUCED_EMBEDDING_DIM)
index.describe_index_stats()

Index 'second-index' already exists → deleting ...
Deleted 'second-index' successfully!
Creating index 'second-index' ...
Index 'second-index' created successfully!


{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [11]:
def upsert_df(df, index, batch_size=128):
    batch_to_upsert = []

    for i, row in tqdm(df.iterrows(), total=df.shape[0], leave=False):
        id = row['id']
        metadata = (
            ast.literal_eval(row['metadata'])
            if isinstance(row['metadata'], str)
            else row['metadata']
        )
        vector = (
            ast.literal_eval(row['values_reduced'])
            if isinstance(row['values_reduced'], str)
            else row['values_reduced']
        )

        batch_to_upsert.append(
            {'id': id, 'values': vector, 'metadata': metadata}
        )

        if len(batch_to_upsert) >= batch_size:
            index.upsert(batch_to_upsert)
            batch_to_upsert = []

    # the last one may not >= batch_size
    if batch_to_upsert:
        index.upsert(batch_to_upsert)

In [12]:
upsert_df(df, index, batch_size=512)

  0%|          | 0/10000 [00:00<?, ?it/s]

In [13]:
index.describe_index_stats()

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 10000}},
 'total_vector_count': 10000,
 'vector_type': 'dense'}

# Prompt

In [14]:
# def get_embeddings(texts, model='text-embedding-ada-002'):
#     openai_client = OpenAI(api_key=Utils.get_openai_api_key())
#     return openai_client.embeddings.create(input=texts, model=model)

**If you using OpenAI model for embedding task, you dont need to reduce dimension of embeddings like I do**

In [15]:
embedding_model = SentenceTransformer('all-mpnet-base-v2')
embedding_model.get_sentence_embedding_dimension()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


768

In [16]:
def search_semantic(text, index, model, k=5):
    vector = model.encode(question).tolist()
    results = index.query(vector=vector, top_k=k, include_metadata=True, include_values=False)

    scores = []
    similar_texts = []

    for match in results['matches']:
        score = match['score']
        meta_text = match['metadata']['text']

        similar_texts.append(meta_text)
        scores.append(score)

    return similar_texts, scores

In [17]:
def build_prompt(question, retrieved_texts, system_prompt=None):
    context_block = "\n".join(
        [f"{i+1}. {t.strip()}" for i, t in enumerate(retrieved_texts) if t.strip()]
    )

    system_prompt = system_prompt or (
        "You are a helpful AI assistant. Use the retrieved context to answer accurately and concisely."
    )

    user_prompt = (
        f"Here are the retrieved documents:\n{context_block}\n\n"
        f"Now answer the question below **only if relevant information is present**.\n"
        f"If not enough information is available, say so.\n\n"
        f"Question: {question}"
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [18]:
question = "What is RAG?"
retrieved_texts = [
    "RAG stands for Retrieval-Augmented Generation.",
    "It combines a retriever that fetches relevant documents with a generator model that forms the final answer."
]

messages = build_prompt(question, retrieved_texts)
pprint.pprint(messages)

[{'content': 'You are a helpful AI assistant. Use the retrieved context to '
             'answer accurately and concisely.',
  'role': 'system'},
 {'content': 'Here are the retrieved documents:\n'
             '1. RAG stands for Retrieval-Augmented Generation.\n'
             '2. It combines a retriever that fetches relevant documents with '
             'a generator model that forms the final answer.\n'
             '\n'
             'Now answer the question below **only if relevant information is '
             'present**.\n'
             'If not enough information is available, say so.\n'
             '\n'
             'Question: What is RAG?',
  'role': 'user'}]


# Chat Model

**Here is the docs: *https://huggingface.co/docs/inference-providers/tasks/chat-completion***

In [19]:
client = InferenceClient(
    token=Utils.get_huggingface_api_key()
)

response = client.chat_completion(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=messages,
    max_tokens=512,
    temperature=0.7,
    top_p=0.95,
)

print(response.choices[0].message["content"])

RAG stands for Retrieval-Augmented Generation. It combines a retriever that fetches relevant documents with a generator model that forms the final answer.


In [20]:
# model can returns multiple choices, we pick the first one to see what's inside
response.choices[0]

ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='RAG stands for Retrieval-Augmented Generation. It combines a retriever that fetches relevant documents with a generator model that forms the final answer.', reasoning=None, tool_call_id=None, tool_calls=[]), logprobs=None, seed=11358461004454160000)

# RAG Pipeline

In [21]:
def generate_answer(question, index, response_model, hf_client, embedding_model, k=5, system_prompt=None):
    retrieved_texts, _ = search_semantic(question, index, embedding_model, k=k)
    messages=build_prompt(question, retrieved_texts, system_prompt)

    response = hf_client.chat_completion(
        model=response_model,
        messages=messages,
        max_tokens=512,
        temperature=0.7,
        top_p=0.9,
    )

    answer = response.choices[0].message["content"]
    print("Answer:\n", answer)

In [22]:
question = "write an article titled: what is the berlin wall?"
_, scores = search_semantic(question, index, embedding_model, k=5)
scores

[0.146718234, 0.120976314, 0.118989721, 0.114585198, 0.113040432]

**Low Score because the embedding model used in dataset is different with embedding model we use to encode question**

In [23]:
generate_answer(
    question, index=index,
    response_model="Qwen/Qwen2.5-7B-Instruct", hf_client=client,
    embedding_model=embedding_model
)

Answer:
 The provided documents do not contain any information about the Berlin Wall. Therefore, I cannot write an article titled "What is the Berlin Wall?" based on the given context.
